# Question 1 - Final Regular Season Ranking

## Objective

The objective is to predict the final regular-season ranking of teams for a selected league and season.

The prediction is made before the target season starts. Therefore, only information from previous seasons can be used.

The modelling pipeline is designed to allow different leagues and target seasons to be selected.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Prediction configuration
LEAGUE_ID = "NHL"
TARGET_YEAR = 2008

In [3]:
def read_and_clean(path):
    df = pd.read_csv(
        path,
        dtype=str,
        skipinitialspace=True
    )

    df.columns = df.columns.str.strip()

    for col in df.columns:
        df[col] = df[col].str.strip()

    return df.replace("", np.nan)

In [4]:
teams = read_and_clean(
    "../hockey_development_data/Teams.csv"
)

scoring = read_and_clean(
    "../hockey_development_data/Scoring.csv"
)

goalies = read_and_clean(
    "../hockey_development_data/Goalies.csv"
)

coaches = read_and_clean(
    "../hockey_development_data/Coaches.csv"
)

## League Data Preparation

Only data from the selected league is used.

Relevant numerical columns are converted to numeric format before creating the modelling features.

In [6]:
# Keep only data from the selected league
teams_league = teams[
    teams["lgID"] == LEAGUE_ID
].copy()

scoring_league = scoring[
    scoring["lgID"] == LEAGUE_ID
].copy()

goalies_league = goalies[
    goalies["lgID"] == LEAGUE_ID
].copy()

coaches_league = coaches[
    coaches["lgID"] == LEAGUE_ID
].copy()

In [7]:
team_numeric_cols = [
    "year",
    "rank",
    "G",
    "W",
    "L",
    "Pts",
    "GF",
    "GA"
]

for col in team_numeric_cols:
    teams_league[col] = pd.to_numeric(
        teams_league[col],
        errors="coerce"
    )

## Additional Team-Level Statistics

Player, goalie and coach information is aggregated to the team-season level.

These statistics provide additional context about offensive production, goalkeeper performance and coaching changes.

In [8]:
# Convert year to numeric
scoring_league["year"] = pd.to_numeric(
    scoring_league["year"],
    errors="coerce"
)

# Convert scoring columns to numeric
scoring_cols = ["G", "A", "Pts", "PIM"]

scoring_league[scoring_cols] = scoring_league[scoring_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# Aggregate player statistics by team and season
team_scoring = (
    scoring_league
    .groupby(["tmID", "year"])[scoring_cols]
    .sum(min_count=1)
    .reset_index()
)

team_scoring = team_scoring.rename(columns={
    "G": "team_player_goals",
    "Pts": "team_player_pts",
    "PIM": "team_player_pim"
})

In [9]:
# Convert year to numeric
goalies_league["year"] = pd.to_numeric(
    goalies_league["year"],
    errors="coerce"
)

# Convert goalie columns to numeric
goalie_cols = ["SA", "GA"]

goalies_league[goalie_cols] = goalies_league[goalie_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# Aggregate goalie statistics by team and season
team_goalies = (
    goalies_league
    .groupby(["tmID", "year"])[goalie_cols]
    .sum(min_count=1)
    .reset_index()
)

# Calculate team save percentage
team_goalies["team_save_pct"] = np.where(
    team_goalies["SA"] > 0,
    1 - (team_goalies["GA"] / team_goalies["SA"]),
    np.nan
)

In [10]:
# Convert year to numeric
coaches_league["year"] = pd.to_numeric(
    coaches_league["year"],
    errors="coerce"
)

# Count the number of coaches used by each team-season
team_coaches = (
    coaches_league
    .groupby(["tmID", "year"])["coachID"]
    .nunique()
    .reset_index(name="num_coaches")
)

In [11]:
df = teams_league.merge(
    team_scoring,
    on=["tmID", "year"],
    how="left"
)

df = df.merge(
    team_goalies[["tmID", "year", "team_save_pct"]],
    on=["tmID", "year"],
    how="left"
)

df = df.merge(
    team_coaches,
    on=["tmID", "year"],
    how="left"
)

## Normalized Performance Features

Raw season totals can be misleading because the number of games played may vary between seasons.

Therefore, the main performance statistics are converted into per-game measures.

In [14]:
# Avoid division by zero
df["G"] = df["G"].replace(0, np.nan)

# Create normalized performance features
df["win_pct"] = (
    df["W"] / df["G"]
)

df["league_pts_per_game"] = (
    df["Pts"] / df["G"]
)

df["goal_diff_per_game"] = (
    (df["GF"] - df["GA"]) / df["G"]
)

df["player_pts_per_game"] = (
    df["team_player_pts"] / df["G"]
)

df["player_goals_per_game"] = (
    df["team_player_goals"] / df["G"]
)

df["player_pim_per_game"] = (
    df["team_player_pim"] / df["G"]
)

## Previous-Season Features

To predict a target season, only information available before that season can be used.

Therefore, the main performance variables are shifted by one season within each franchise.

In [16]:
df = df.sort_values(
    ["franchID", "year"]
).reset_index(drop=True)

In [26]:
previous_cols = [
    "league_pts_per_game",
    "goal_diff_per_game",
    "team_save_pct",
    "player_pts_per_game",
    "num_coaches",
    "rank"
]

for col in previous_cols:
    df[f"prev_{col}"] = (
        df.groupby("franchID")[col]
        .shift(1)
    )

In [27]:
# Performance from two seasons earlier
df["prev_2y_league_pts_per_game"] = (
    df.groupby("franchID")["league_pts_per_game"]
    .shift(2)
)

# Recent performance trend
df["pts_momentum"] = (
    df["prev_league_pts_per_game"]
    - df["prev_2y_league_pts_per_game"]
)

## Target and Train-Test Split

The target is the performance of each team in the selected season.

All seasons before the target year are used for training, while the selected target year is used for testing.

In [28]:
# Define target
df["target"] = df["league_pts_per_game"]
df["target_rank"] = df["rank"]

# Split chronologically
train = df[df["year"] < TARGET_YEAR].copy()
test = df[df["year"] == TARGET_YEAR].copy()

In [29]:
features = [
    "prev_league_pts_per_game",
    "prev_goal_diff_per_game",
    "prev_team_save_pct",
    "prev_player_pts_per_game",
    "prev_num_coaches",
    "prev_rank",
    "pts_momentum"
]

In [30]:
train_medians = train[features].median()

train[features] = (
    train[features]
    .fillna(train_medians)
    .fillna(0)
)

test[features] = (
    test[features]
    .fillna(train_medians)
    .fillna(0)
)

In [31]:
X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

## Model Comparison for Selected Season

Several regression models are trained using the same features.

The models predict team performance for the target season.  
The predicted performance is then converted into rankings within each division and compared with the real ranking.

In [32]:
models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    )
}

In [33]:
results = []
predictions = {}

for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predict target season
    pred_test = model.predict(X_test)

    predictions[name] = pred_test

    # Regression metrics
    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred_test
        )
    )

    r2 = r2_score(
        y_test,
        pred_test
    )

    # Convert predicted performance into division ranking
    temp = test[
        ["divID", "target_rank"]
    ].copy()

    temp["predicted_performance"] = pred_test

    temp["predicted_rank"] = (
        temp.groupby("divID")["predicted_performance"]
        .rank(
            ascending=False,
            method="first"
        )
        .astype(int)
    )

    # Mean absolute ranking error
    rank_error = (
        temp["target_rank"]
        - temp["predicted_rank"]
    ).abs().mean()

    results.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2,
        "Mean Rank Error": rank_error
    })

In [35]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    "Mean Rank Error"
)

,Model,RMSE,R2,Mean Rank Error
0,Linear Regression,0.162240,0.041437,0.866667
2,Random Forest,0.165447,0.003173,0.933333
1,KNN Regressor,0.188393,-0.292506,1.133333
3,Gradient Boosting,0.176241,-0.131133,1.266667


## Selected Season Ranking

The model with the lowest ranking error for the selected season is used to display the predicted ranking.

This section is used to inspect the predictions team by team for the chosen target season.

In [36]:
# Select best model
best_model_name = (
    results_df
    .sort_values("Mean Rank Error")
    .iloc[0]["Model"]
)

test_results = test[
    ["year", "tmID", "name", "divID", "target_rank"]
].copy()

test_results["predicted_performance"] = predictions[
    best_model_name
]

test_results["predicted_rank"] = (
    test_results
    .groupby("divID")["predicted_performance"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

test_results["rank_error"] = (
    test_results["target_rank"]
    - test_results["predicted_rank"]
).abs()

In [38]:
test_results.sort_values(
    ["divID", "predicted_rank"]
)

,year,tmID,name,divID,target_rank,predicted_performance,predicted_rank,rank_error
993,2008,PIT,Pittsburgh Penguins,AT,2,1.127468,1,1
736,2008,NJD,New Jersey Devils,AT,1,1.102850,2,1
856,2008,NYR,New York Rangers,AT,4,1.086471,3,1
915,2008,PHI,Philadelphia Flyers,AT,3,1.018259,4,1
773,2008,NYI,New York Islanders,AT,5,1.009155,5,0
486,2008,DET,Detroit Red Wings,CE,1,1.237831,1,0
701,2008,NAS,Nashville Predators,CE,5,1.070204,2,3
320,2008,CHI,Chicago Blackhawks,CE,2,1.002186,3,1
1070,2008,STL,St. Louis Blues,CE,3,0.960205,4,1
237,2008,CBS,Columbus Blue Jackets,CE,4,0.952969,5,1


## Evaluation Across Multiple Seasons

To choose a final model, the algorithms are evaluated across several historical seasons.

This avoids selecting a model based on a single target year and provides a more reliable comparison of generalization performance.

### Historical Evaluation Setup

Each season is predicted using only seasons that occurred before it.

Very early seasons contain too little historical data to train a meaningful model. Therefore, a season is evaluated only when at least 50 previous team-season observations are available for training.

In [39]:
MIN_TRAIN_SAMPLES = 50

In [40]:
def evaluate_year(target_year):

    # Split training and test data
    train = df[df["year"] < target_year].copy()
    test = df[df["year"] == target_year].copy()

    # Ignore seasons with too little training data
    if len(train) < MIN_TRAIN_SAMPLES:
        return pd.DataFrame()

    # Fill missing values using training medians
    train_medians = train[features].median()

    train[features] = (
        train[features]
        .fillna(train_medians)
        .fillna(0)
    )

    test[features] = (
        test[features]
        .fillna(train_medians)
        .fillna(0)
    )

    # Prepare training and test sets
    X_train = train[features]
    y_train = train["target"]

    X_test = test[features]
    y_test = test["target"]

    n_train = len(X_train)

    year_results = []

    for name, model in models.items():

        # Adapt KNN when the training set is small
        if name == "KNN Regressor":
            model = KNeighborsRegressor(
                n_neighbors=min(5, n_train)
            )

        # Train model
        model.fit(X_train, y_train)

        # Predict target season
        pred_test = model.predict(X_test)

        # Regression metrics
        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred_test
            )
        )

        r2 = r2_score(
            y_test,
            pred_test
        )

        # Prepare ranking evaluation
        temp = test[
            ["divID", "target_rank"]
        ].copy()

        temp["predicted_performance"] = pred_test

        # Rank within divisions when divisions exist
        if test["divID"].notna().any():
            temp["predicted_rank"] = (
                temp
                .groupby("divID")["predicted_performance"]
                .rank(
                    ascending=False,
                    method="first"
                )
            )

        # Otherwise rank within the whole league
        else:
            temp["predicted_rank"] = (
                temp["predicted_performance"]
                .rank(
                    ascending=False,
                    method="first"
                )
            )

        # Mean absolute ranking error
        rank_error = (
            temp["target_rank"]
            - temp["predicted_rank"]
        ).abs().mean()

        year_results.append({
            "Year": target_year,
            "Model": name,
            "RMSE": rmse,
            "R2": r2,
            "Mean Rank Error": rank_error
        })

    return pd.DataFrame(year_results)

In [41]:
years_to_test = sorted(
    df.loc[
        df["year"] <= 2009,
        "year"
    ].dropna().unique()
)

# Remove the first season because there is no previous-season information
years_to_test = years_to_test[1:]

In [45]:
all_results = pd.concat(
    [
        evaluate_year(year)
        for year in years_to_test
    ],
    ignore_index=True
)

years_to_show = [
    1927,
    1940,
    1960,
    1980,
    1990,
    2000,
    2005,
    2008,
    2009
]

all_results[
    all_results["Year"].isin(years_to_show)
]

,Year,Model,RMSE,R2,Mean Rank Error
0,1927,Linear Regression,0.320065,-0.408571,1.400000
1,1927,KNN Regressor,0.280687,-0.083294,1.200000
2,1927,Random Forest,0.273245,-0.026615,1.600000
3,1927,Gradient Boosting,0.304046,-0.271102,1.800000
52,1940,Linear Regression,0.196461,0.484690,1.142857
53,1940,KNN Regressor,0.214929,0.383256,1.428571
54,1940,Random Forest,0.236561,0.252865,1.428571
55,1940,Gradient Boosting,0.209007,0.416778,0.571429
132,1960,Linear Regression,0.196624,0.404903,0.666667
133,1960,KNN Regressor,0.182770,0.485810,0.666667


In [47]:
model_summary = (
    all_results
    .groupby("Model")
    .agg({
        "Mean Rank Error": "mean",
        "RMSE": "mean",
        "R2": "mean"
    })
    .sort_values("Mean Rank Error")
)

model_summary

,Mean Rank Error,RMSE,R2
Model,,,
Linear Regression,1.068857,0.175321,0.337707
Random Forest,1.150240,0.185120,0.233969
Gradient Boosting,1.155108,0.188672,0.222600
KNN Regressor,1.194212,0.197576,0.117117


## Baseline Model Selection

The models were evaluated across all valid historical seasons with at least 50 training observations.

Linear Regression achieved the best overall performance, with the lowest mean ranking error and the highest average R².

Therefore, Linear Regression is selected as the baseline model for Question 1.

Further improvements will be compared against this baseline.

## Feature Selection Experiments

Several feature experiments were performed using the same historical evaluation procedure.

- **TeamVsTeam features** were tested but did not improve the baseline, so they were discarded.
- **Relative season strength** produced only a very small improvement and was not kept.
- **Previous-season penalty minutes (`prev_player_pim_per_game`)** were removed because the historical mean rank error improved.
- **Previous-season number of coaches (`prev_num_coaches`)** was also removed because it did not improve the model.
- **Previous-season player points per game (`prev_player_pts_per_game`)** was tested for removal, but performance became worse, so this feature was kept.
- **Recent performance momentum (`pts_momentum`)** improved the historical mean rank error and was therefore added to the model.

After these experiments, Linear Regression remained the best-performing model.